# 📖 Notebook 10: Production Patterns — Scaling, Multi-Tenancy, and Disaster Recovery

Welcome to the final notebook in the Kubernetes lab series. Up to this point, you have learned how to build, expose, secure, observe, and persist workloads. Now we step into the production mindset: how do you keep systems available, efficient, isolated, and recoverable when real traffic and real failures show up?


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain **resource requests** and **resource limits**
- Recognize Kubernetes **QoS classes**: Guaranteed, Burstable, and BestEffort
- Use a **HorizontalPodAutoscaler (HPA)** to scale based on usage
- Explain when **Vertical Pod Autoscaler (VPA)** is useful
- Create a **PodDisruptionBudget (PDB)** for critical workloads
- Describe common **multi-tenancy** patterns using namespaces, quotas, and policies
- Explain how **Velero** helps with backup and restore
- Build a simple production readiness checklist for Kubernetes services


## 🛠️ Setup

Before you begin:

- Start Minikube and make sure the `k8s-lab` namespace is still available
- Run this lab from `03-technologies/container-orchestration/kubernetes/` so the reference manifests are easy to find
- Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

This notebook assumes the sample FastAPI microservices are already running:

- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

We also need the Kubernetes metrics pipeline, because HPA depends on usage metrics such as CPU.


In [ ]:
!kubectl config current-context
!kubectl get ns k8s-lab
!kubectl get pods -n k8s-lab
!minikube addons enable metrics-server
!kubectl get deployment metrics-server -n kube-system
!kubectl top pods -n k8s-lab || echo 'Metrics may take a minute to appear after enabling metrics-server.'


## Production vs Development

A development cluster proves that your app can run. A production cluster must prove that your app can survive.

In development, you might be okay with:

- one replica
- weak defaults
- manual restarts
- no backup story

In production, you care about:

- predictable resource usage
- automatic scaling
- safe maintenance windows
- tenant isolation
- disaster recovery

This notebook is about those patterns.


## 📦 Resource Requests and Limits

Every container competes for CPU and memory on a node. Kubernetes needs hints so it can schedule workloads safely.

- **Request** = what the container is guaranteed to get
- **Limit** = the maximum the container is allowed to use

Think of requests as a reserved seat on a train, and limits as the maximum luggage allowance.

These settings also shape the pod's **QoS class**:

- **Guaranteed**: requests and limits are set and equal for CPU and memory
- **Burstable**: some requests/limits exist, but they are not equal
- **BestEffort**: no requests and no limits

### ✅ Exercise
Create one pod for each QoS class and inspect how Kubernetes labels them.


In [ ]:
%%writefile qos-demo.yaml
apiVersion: v1
kind: Pod
metadata:
  name: guaranteed-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      resources:
        requests:
          cpu: 100m
          memory: 128Mi
        limits:
          cpu: 100m
          memory: 128Mi
---
apiVersion: v1
kind: Pod
metadata:
  name: burstable-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      resources:
        requests:
          cpu: 100m
          memory: 64Mi
        limits:
          cpu: 300m
          memory: 256Mi
---
apiVersion: v1
kind: Pod
metadata:
  name: besteffort-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]


In [ ]:
!kubectl delete pod guaranteed-demo burstable-demo besteffort-demo -n k8s-lab --ignore-not-found
!kubectl apply -f qos-demo.yaml
!kubectl wait --for=condition=Ready pod/guaranteed-demo -n k8s-lab --timeout=120s
!kubectl wait --for=condition=Ready pod/burstable-demo -n k8s-lab --timeout=120s
!kubectl wait --for=condition=Ready pod/besteffort-demo -n k8s-lab --timeout=120s
!kubectl get pods -n k8s-lab -o custom-columns=NAME:.metadata.name,QOS:.status.qosClass


## 🔁 HorizontalPodAutoscaler (HPA)

An **HPA** means: **more pods when busy, fewer pods when idle**.

Instead of making one pod bigger, HPA adds or removes replicas. This usually works very well for stateless services like our `api-gateway`.

We already have an HPA manifest in this lab series. We will apply it, generate load, and watch Kubernetes react.

### ✅ Exercise
Apply the HPA, generate traffic, and observe the replica count change.


In [ ]:
!kubectl apply -f manifests/hpa.yaml || kubectl apply -f ../manifests/hpa.yaml
!kubectl get hpa api-gateway-hpa -n k8s-lab
!kubectl get deployment api-gateway -n k8s-lab


In [ ]:
!kubectl delete pod load-gen -n k8s-lab --ignore-not-found
!kubectl run load-gen -n k8s-lab --image=busybox -- /bin/sh -c "while true; do wget -q -O- http://api-gateway.k8s-lab:8000/health; done"


In [ ]:
!kubectl get hpa -n k8s-lab -w


When you have seen the HPA change, interrupt the cell above, then run the next cell to stop the synthetic traffic and watch the scale-down behavior.


In [ ]:
!kubectl delete pod load-gen -n k8s-lab --ignore-not-found
!kubectl get hpa api-gateway-hpa -n k8s-lab
!kubectl get deployment api-gateway -n k8s-lab


## 📈 Vertical Pod Autoscaler (VPA) — Concept Only

If HPA means **more pods**, VPA means **bigger or smaller pods**. A VPA watches usage and recommends or sets new CPU and memory requests.

Use cases:

- **HPA** is great when the workload scales horizontally
- **VPA** is useful when a workload cannot easily be split into more replicas or when you want better sizing recommendations

A simple rule of thumb:

- web APIs: usually HPA first
- singleton workloads, batch jobs, or memory-heavy services: VPA may help

### ✅ Exercise
Look at `api-gateway`, `user-service`, and `order-service`. Which one would you scale horizontally first? Which one might benefit from better vertical sizing? Write down your answer before moving on.


## 🛡️ PodDisruptionBudget

A **PodDisruptionBudget (PDB)** tells Kubernetes: *during voluntary disruptions, always keep at least this many pods available*.

This matters during things like:

- node maintenance
- cluster upgrades
- draining a node

A PDB does **not** stop crashes or involuntary failures. It protects you during planned operations.

### ✅ Exercise
Create a PDB for `api-gateway` with `minAvailable: 1` and inspect it. Then review the drain command you would use during maintenance.


In [ ]:
%%writefile api-gateway-pdb.yaml
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: api-gateway-pdb
  namespace: k8s-lab
spec:
  minAvailable: 1
  selector:
    matchLabels:
      app: api-gateway


In [ ]:
!kubectl apply -f api-gateway-pdb.yaml
!kubectl get pdb -n k8s-lab
!kubectl get nodes


In [ ]:
!echo 'Optional maintenance test:'
!echo 'kubectl drain minikube --ignore-daemonsets --delete-emptydir-data --force'
!echo 'kubectl uncordon minikube'


## 🏢 Multi-Tenancy

**Multi-tenancy** means multiple teams or applications share the same cluster without stepping on each other.

A common beginner-friendly pattern is **namespace-per-team**. Each team gets:

- its own namespace
- its own ResourceQuota
- its own LimitRange defaults
- its own NetworkPolicy rules

This is not perfect isolation like separate clusters, but it is a very common and practical starting point.

### ✅ Exercise
Create `team-a` and `team-b` namespaces with quotas, default limits, and a default-deny NetworkPolicy.


In [ ]:
%%writefile team-tenancy.yaml
apiVersion: v1
kind: Namespace
metadata:
  name: team-a
---
apiVersion: v1
kind: Namespace
metadata:
  name: team-b
---
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-a-quota
  namespace: team-a
spec:
  hard:
    pods: "5"
    requests.cpu: "1"
    requests.memory: 1Gi
    limits.cpu: "2"
    limits.memory: 2Gi
---
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-b-quota
  namespace: team-b
spec:
  hard:
    pods: "5"
    requests.cpu: "1"
    requests.memory: 1Gi
    limits.cpu: "2"
    limits.memory: 2Gi
---
apiVersion: v1
kind: LimitRange
metadata:
  name: team-a-defaults
  namespace: team-a
spec:
  limits:
    - type: Container
      default:
        cpu: 500m
        memory: 256Mi
      defaultRequest:
        cpu: 100m
        memory: 128Mi
---
apiVersion: v1
kind: LimitRange
metadata:
  name: team-b-defaults
  namespace: team-b
spec:
  limits:
    - type: Container
      default:
        cpu: 500m
        memory: 256Mi
      defaultRequest:
        cpu: 100m
        memory: 128Mi
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny
  namespace: team-a
spec:
  podSelector: {}
  policyTypes:
    - Ingress
    - Egress
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny
  namespace: team-b
spec:
  podSelector: {}
  policyTypes:
    - Ingress
    - Egress


In [ ]:
!kubectl apply -f team-tenancy.yaml
!kubectl get resourcequota -A
!kubectl get limitrange -A
!kubectl get networkpolicy -A


## 💽 Velero Backup and Restore

Backups are a production feature, not an afterthought. A Kubernetes backup strategy usually needs two things:

1. cluster resources such as Deployments, Services, and ConfigMaps
2. persistent data from volumes

**Velero** is a popular tool for this. It can back up Kubernetes objects and, depending on your storage setup, snapshot or copy persistent volume data too.

In a real environment, Velero also needs an object storage destination and credentials. We will keep this section conceptual but use real commands so you can see the workflow.

### ✅ Exercise
Add the Helm repo, review the chart, and read the backup, restore, and schedule commands below.


In [ ]:
!helm repo add vmware-tanzu https://vmware-tanzu.github.io/helm-charts
!helm repo update
!helm search repo vmware-tanzu/velero


In [ ]:
!velero backup create k8s-lab-backup --include-namespaces k8s-lab
!velero restore create --from-backup k8s-lab-backup
!velero schedule create daily-k8s-lab --schedule="0 2 * * *" --include-namespaces k8s-lab


## ✅ Production Checklist

Before calling a service production-ready, walk through this checklist:

- Resource requests and limits on every container
- Readiness and liveness probes
- PodDisruptionBudget for critical services
- HPA for variable-load services
- NetworkPolicy with a default-deny posture
- RBAC with least privilege
- Secrets from an external store when possible
- GitOps for deployment and drift correction
- Monitoring and alerting
- A tested backup and restore strategy

If a team cannot explain how it handles each item above, it is not really production-ready yet.


## 🧹 Clean Up the Entire Lab

This command deletes the whole Minikube cluster. Use it when you are done with the series and want a completely clean slate.


In [ ]:
!minikube delete


## 🎓 What You Learned

You made it to the end of the 10-notebook Kubernetes lab series. Here is the big picture:

1. You created and explored a cluster
2. You deployed pods and deployments
3. You exposed services and networking paths
4. You packaged apps with Helm and Kustomize
5. You added observability
6. You locked things down with RBAC and network policies
7. You used GitOps ideas with ArgoCD
8. You explored service mesh concepts
9. You added storage and secrets
10. You learned production patterns for scaling, isolation, and recovery

That is a strong beginner-to-intermediate foundation. You now understand not just how to run containers in Kubernetes, but how to think like a platform engineer.


## 🚀 Where to Go Next

If you want to keep going, here are great next topics:

- Kubernetes official docs: workloads, networking, storage, and security
- Learn a GitOps workflow deeply with ArgoCD or Flux
- Practice production observability with Prometheus, Grafana, and Alertmanager
- Explore cluster security tools such as Kyverno, Falco, and image scanning
- Study platform engineering topics such as internal developer platforms and golden paths
- Try the same lab ideas on a managed cloud cluster such as AKS, EKS, or GKE

Most importantly: keep practicing. Kubernetes becomes much easier once you have seen the same ideas from the app side, the platform side, and the operations side.
